<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Задача 2

*Будем тестировать CLIP на задаче классификации. Для этого нужно взять готовый датасет imagenette (https://docs.pytorch.org/vision/main/datasets.html) (можно подкачать из библиотеки torchvision) и протестировать на нем CLIP (а по какой метрике тестировать? А какие бывают? Надо погуглить).
Замечание: imagenette состоящий из 10 классов - это подмножество большого ImageNet1k, в котором 1000 классов.*

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import gc
import numpy as np

In [ ]:
BATCH_SIZE = 128

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to('cuda')
model.eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=False,
                                          num_workers=2,
                                          collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

In [ ]:
classes_for_clip = ['A picture of ' + i[0] for i in imagenette_data.classes]

In [ ]:
correct_classifications = 0
all_classifications = 0
predictions = []
truth = []
cached_text_embeddings = None

txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to('cuda')
text_features = model.get_text_features(**txt_inputs)
text_emb = text_features.pooler_output.detach().to('cuda')
text_emb = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)

for batch, t in tqdm(data_loader):
  with torch.inference_mode():
    img_inputs = processor(images=batch, padding=True, return_tensors='pt').to('cuda')
    image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to('cuda')
    image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)

    logits = image_emb @ text_emb.T
    predicted_cat = logits.argmax(dim=1).to('cpu')

    batch_truth = torch.Tensor(t).to('cpu')

    predictions += predicted_cat.tolist()
    truth += t

    correct_classifications += int((predicted_cat == batch_truth).sum())
    all_classifications += BATCH_SIZE

In [ ]:
freqs = Counter(truth)
plt.title('Imagenette class sizes')
plt.bar(freqs.keys(), freqs.values())

In [ ]:
print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')    # Actually I have no idea in which case we may need micro-averaging.
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')       # The classes are distributed rather equally, so we could use micro,
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')           # but macro seems to be at least not worse.